# M1: T4, Qwen, изолированные тесты и короткий resume

Выберите GPU runtime. На Mac должен работать Docker worker с проверенной изоляцией и общей синхронизируемой папкой. Этот notebook делает только установку и вызовы CLI. Он остановится при неудачном preflight или недоступном worker. Проверка файла очереди обязательна до загрузки Qwen.

SFT и GRPO ограничены двумя optimizer step. Сохранение после первого шага и продолжение выполняются разными процессами. Метрики такого прогона не доказывают качество модели. Инструкции worker и ограничения: `docs/m1.md`.


In [ ]:
REPO_URL = "https://github.com/matveylogee/qwen-repair.git"
COMMIT_SHA = ""  # Full commit SHA containing M1.
CHECKOUT_DIR = "/content/qwen-repair-m1"
ENV_DIR = "/content/qwen-repair-m1-env"
QUEUE_DIR = "/content/drive/MyDrive/qwen-repair-exchange"
MOUNT_GOOGLE_DRIVE = True


In [ ]:
import re
import subprocess
import sys
from pathlib import Path
from urllib.parse import urlparse
from uuid import uuid4

repo = urlparse(REPO_URL)
if repo.scheme != "https" or not repo.hostname or repo.username or repo.password:
    raise ValueError("Use an HTTPS repository URL without credentials.")
if re.fullmatch(r"[0-9a-fA-F]{40}", COMMIT_SHA) is None:
    raise ValueError("Set the full immutable commit SHA before running.")
checkout = Path(CHECKOUT_DIR)
environment = Path(ENV_DIR)
if checkout.exists() or environment.exists():
    raise FileExistsError("Choose new CHECKOUT_DIR and ENV_DIR to preserve earlier runs.")
subprocess.run(["git", "clone", "--", REPO_URL, str(checkout)], check=True)
subprocess.run(["git", "checkout", "--detach", COMMIT_SHA], cwd=checkout, check=True)
actual_sha = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=checkout, text=True).strip()
if actual_sha.lower() != COMMIT_SHA.lower():
    raise RuntimeError("Checkout does not match the requested commit.")
subprocess.run([sys.executable, "-m", "venv", str(environment)], check=True)
python = str(environment / "bin" / "python")
subprocess.run([python, "-m", "pip", "install", "-e", ".[gpu]"], cwd=checkout, check=True)
run_root = checkout / "runs" / f"m1-{uuid4().hex}"
run_root.mkdir(parents=True)
cli = [python, "-m", "qwen_repair"]


In [ ]:
subprocess.run(cli + ["doctor", "--gpu", "--output", str(run_root / "gpu-preflight.json")], cwd=checkout, check=True)


Перед следующей ячейкой запустите `qrepair queue-worker` на Mac из той же версии M1, которая указана в `COMMIT_SHA`. После обновления кода перезапустите worker. Папка Google Drive на Mac должна синхронизироваться с `QUEUE_DIR` в том же аккаунте. При ошибке queue probe не переходите к inference или обучению.


In [ ]:
if MOUNT_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

import json
queue_config = run_root / "queue.yaml"
subprocess.run([
    python, "-c",
    "import pathlib,sys,yaml; p=yaml.safe_load(pathlib.Path(sys.argv[1]).read_text()); p['queue_dir']=sys.argv[2]; pathlib.Path(sys.argv[3]).write_text(yaml.safe_dump(p))",
    str(checkout / "configs/colab_queue.yaml"), QUEUE_DIR, str(queue_config),
], check=True)
subprocess.run(cli + ["queue-probe", "--config", str(queue_config), "--output", str(run_root / "queue-probe.json")], cwd=checkout, check=True)
print((run_root / "queue-probe.json").read_text())


In [ ]:
base_run = run_root / "base"
subprocess.run(cli + ["infer", "--config", "configs/m1_infer.yaml", "--executor-config", str(queue_config), "--output", str(base_run)], cwd=checkout, check=True)
subprocess.run(cli + ["summarize", "--run", str(base_run)], cwd=checkout, check=True)


SFT: первый процесс делает один шаг при запланированных двух; второй загружает checkpoint и заканчивает тот же план. Восстанавливаются optimizer, scheduler и RNG. Это проверка продолжения, а не два независимых запуска.


In [ ]:
sft_run = run_root / "sft"
subprocess.run(cli + ["sft-smoke", "--config", "configs/m1_sft.yaml", "--executor-config", str(queue_config), "--output", str(sft_run), "--stop-after", "1"], cwd=checkout, check=True)


In [ ]:
subprocess.run(cli + ["sft-smoke", "--config", "configs/m1_sft.yaml", "--executor-config", str(queue_config), "--output", str(sft_run), "--resume-from", str(sft_run / "checkpoint-1")], cwd=checkout, check=True)
print((sft_run / "training_result.json").read_text())


GRPO: каждый rollout содержит две попытки, в группе два эпизода одной задачи. Feedback исключён из loss, private reward относится только к последнему кандидату. При одинаковых rewards градиент от reward может быть нулевым; это фиксируется в отчёте. Для этой проверки KL отключён явно (`beta=0`).


In [ ]:
grpo_config = run_root / "grpo.yaml"
subprocess.run([
    python, "-c",
    "import pathlib,sys,yaml; p=yaml.safe_load(pathlib.Path(sys.argv[1]).read_text()); p['sft_adapter']=sys.argv[2]; pathlib.Path(sys.argv[3]).write_text(yaml.safe_dump(p))",
    str(checkout / "configs/m1_grpo.yaml"), str(sft_run / "adapter"), str(grpo_config),
], check=True)
grpo_run = run_root / "grpo"
subprocess.run(cli + ["grpo-smoke", "--config", str(grpo_config), "--executor-config", str(queue_config), "--output", str(grpo_run), "--stop-after", "1"], cwd=checkout, check=True)


In [ ]:
subprocess.run(cli + ["grpo-smoke", "--config", str(grpo_config), "--executor-config", str(queue_config), "--output", str(grpo_run), "--resume-from", str(grpo_run / "checkpoint-1")], cwd=checkout, check=True)
print((grpo_run / "training_result.json").read_text())


Сохраните результаты до завершения runtime. Следующая ячейка упакует весь каталог запуска в ZIP и начнёт скачивание через Colab. Inference bundles, training manifests, checkpoint completion markers и контрольные суммы входят в архив. Checkpoints нужны для восстановления после потери runtime. При ошибке предыдущей ячейки также выполните упаковку, чтобы сохранить имеющиеся отчёты для диагностики.

Если браузер заблокировал скачивание, скачайте напечатанный ZIP через панель Files. Не завершайте runtime до сохранения архива на компьютере.


In [ ]:
import shutil
from google.colab import files

archive = shutil.make_archive(str(run_root), "zip", root_dir=run_root.parent, base_dir=run_root.name)
print(f"Archive: {archive}")
files.download(archive)
